# Глава 7 — Evaluating Agents

Оценка отвечает на разные вопросы: **верен ли ответ**, **правильно ли агент действовал**, **повторяется ли успех**. В этой главе добавляем `Benchmark` и `Evaluator`, три способа проверки результата, а также метрики `pass@k` и `pass^k`.

Этот notebook запускает **маленькие учебные наборы из главы**, а не официальные MMLU-Pro / IFEval. Три примера не позволяют сравнивать модели или делать выводы об общей надёжности. Ожидаемые ответы взяты из примеров книги; это ключи этих заданий, а не независимо проверенная база знаний.

По умолчанию используется установленная `gemma4:e4b` через локальный Ollama. Внешних API, ключей и автоматических загрузок моделей нет.

In [ ]:
from pprint import pprint
import socket

from agent import TinyAgent
from evaluator import (
    Benchmark, Evaluator, exact_match_scorer, programmatic_scorer,
    make_judge_scorer, pass_at_k, pass_hat_k,
)
from llm import LLM
from memory import Memory
from planning import NativeReAct
from tools import NativeTools

socket.setdefaulttimeout(180)
MODEL = "gemma4:e4b"
BASE_URL = "http://localhost:11434/v1"
llm = LLM(MODEL, base_url=BASE_URL, think=True, temperature=0)

def create_agent():
    # The model client can be shared; memory and trajectory must be new per example.
    return TinyAgent(llm, memory=Memory(), tools=NativeTools(), planner=NativeReAct())

evaluator = Evaluator(create_agent)

## 1. Exact match: один правильный вариант

Scorer ожидает ровно одну букву A–J; регистр и внешние пробелы не учитываются. В отличие от поиска первой подходящей буквы в тексте, объяснение «The answer is J» не пройдёт условие «только буква». Неправильный формат и неправильный вариант одинаково дают `False`, поэтому важно смотреть и на `prediction`.

Каждый пример получает собственные память и траекторию: ответ на предыдущий вопрос не помогает следующему.

In [ ]:
mmlu_pro = Benchmark(
    name="MMLU-Pro: 3 chapter examples, strict letter match",
    examples=[
        {
            "task": """Which of the following is the body cavity that contains the pituitary gland?
A) Ventral B) Dorsal C) Buccal D) Thoracic E) Pericardial F) Abdominal
G) Spinal H) Pelvic I) Pleural J) Cranial
Answer with only the letter.""",
            "expected": "J",
        },
        {
            "task": """What is the approximate mean cranial capacity of Homo erectus?
A) 1200 cc B) under 650 cc C) 1700 cc D) 1350 cc E) just under 1000 cc
F) 1500 cc G) under 500 cc H) about 800 cc I) just over 1100 cc J) about 900 cc
Answer with only the letter.""",
            "expected": "E",
        },
        {
            "task": """According to Moore's 'ideal utilitarianism,' the right action is the one
that brings about the greatest amount of:
A) wealth B) virtue C) fairness D) pleasure E) peace F) justice
G) happiness H) power I) good J) knowledge
Answer with only the letter.""",
            "expected": "I",
        },
    ],
    scorer=exact_match_scorer,
)
exact_result = evaluator.run(mmlu_pro)
pprint(exact_result)

## 2. Programmatic checks: проверяем конкретное условие

Примеры главы используют простые эвристики: число знаков `. ! ?`, число элементов `split()` и отсутствие запятой. Это **не официальные валидаторы IFEval**. Они не проверяют юмор, полезность маршрута или шекспировский стиль; счётчик пунктуации также не является полноценным разбиением на предложения.

Пустой ответ никогда не проходит. Даже результат `1.0` говорит только о проверенных ограничениях.

In [ ]:
ifeval = Benchmark(
    name="IFEval: 3 simplified chapter checks",
    examples=[
        {
            "task": "Write me a funny song with less than 10 sentences for a proposal to build a new playground at my local elementary school.",
            "check": lambda text: sum(1 for c in text if c in ".!?") < 10,
        },
        {
            "task": "Write an ad copy for a new product, a digital photo frame that connects to your social media accounts and displays your photos. Respond with at most 150 words.",
            "check": lambda text: len(text.split()) <= 150,
        },
        {
            "task": "I am planning a trip to Japan, and I would like thee to write an itinerary for my journey in a Shakespearean style. You are not allowed to use any commas in your response.",
            "check": lambda text: "," not in text,
        },
    ],
    scorer=programmatic_scorer,
)
programmatic_result = evaluator.run(ifeval)
pprint(programmatic_result)

## 3. LLM-as-a-judge: оцениваем смысл ответа

Судья получает вопрос, эталон и ответ кандидата. Его шкала: `0` — неверно, `0.5` — частично верно, `1` — полностью верно. Допускаются и промежуточные оценки.

В книге судья — внешняя Gemini. Здесь используется **та же локальная модель**, чтобы пример запускался без внешнего API. Это демонстрация механизма: общие ошибки модели и предпочтение собственных ответов могут завышать оценку. Для независимой оценки нужен отдельно выбранный судья и проверка его согласия с человеком.

Данные отделены от системной инструкции, но это не гарантирует устойчивость судьи к prompt injection. Некорректный числовой ответ судьи вызывает исключение: мы не угадываем оценку и не заменяем ошибку нулём.

In [ ]:
judge = LLM(MODEL, base_url=BASE_URL, think=False, temperature=0)
judged_benchmark = Benchmark(
    name="Open-ended chapter examples: local same-model judge",
    examples=[
        {"task": "Which body cavity contains the pituitary gland?", "expected": "the cranial cavity"},
        {"task": "What is the approximate mean cranial capacity of Homo erectus?", "expected": "just under 1000 cc"},
        {"task": "According to Moore's 'ideal utilitarianism,' the right action is the one that brings about the greatest amount of what?", "expected": "good"},
    ],
    scorer=make_judge_scorer(judge),
)
judge_result = evaluator.run(judged_benchmark)
pprint(judge_result)
print("Средняя оценка судьи:", judge_result["pass_rate"])

Поле `pass_rate` сохранено как в книге. Для boolean-scorer это доля пройденных задач; для дробных оценок — **средний балл**, а не доля успешных ответов. Значения нельзя сравнивать как одну и ту же метрику.

`completed=False` означает, что агент вернул observation или исчерпал лимит без финального ответа. Такой запуск получает ноль, даже если строка ошибки формально удовлетворяет ограничению по длине. Пустой финальный ответ также получает ноль. Ошибки сети, кода scorer или формата оценки судьи прерывают оценку исключением.

In [ ]:
for result in (exact_result, programmatic_result, judge_result):
    assert len(result["results"]) == 3
    assert 0 <= result["pass_rate"] <= 1
    assert all(0 <= row["passed"] <= 1 for row in result["results"])
    print(result["name"], "→", result["pass_rate"])
# These checks validate the report, not a required quality score from the model.

## 4. Capability и reliability — разные вопросы

- **pass@k:** есть ли хотя бы один успех среди `k` попыток?
- **pass^k:** успешны ли все `k` попыток?

Обе функции оценивают результат выбора `k` попыток без возвращения из `n` наблюдавшихся запусков **одного задания**, где `c` успешны. При `k=1` обе равны `c/n`. Нельзя подставлять сюда средний балл судьи или объединять разные задания в один пул попыток.

Ниже синтетические `6` успехов из `10`, а не измерение Gemma. Для реальной оценки нужны повторные независимые запуски одного задания с бинарной проверкой. Маленький набор и особенно `k`, близкое к `n`, не дают надёжного вывода о будущих запусках.

In [ ]:
n_samples, n_correct_samples = 10, 6
for k in (1, 2, 3):
    print(f"k={k}: pass@k={pass_at_k(n_samples, n_correct_samples, k):.3f}; "
          f"pass^k={pass_hat_k(n_samples, n_correct_samples, k):.3f}")
assert abs(pass_at_k(10, 6, 3) - 29 / 30) < 1e-12
assert abs(pass_hat_k(10, 6, 3) - 1 / 6) < 1e-12

## Что ещё оценивает глава

**Траектория:** правильный ответ не показывает, использовал ли агент нужные инструменты и сколько шагов потратил. В главе 6 мы проверяли реальные `add → multiply → subtract`; здесь `Evaluator` оценивает именно итог. Отдельный scorer траекторий не добавлен.

**Rubric и человек:** критерии стоит задавать отдельно — правильность, полнота, подтверждение источниками. Мнение LLM-судьи нужно сопоставлять с человеческой оценкой.

**Safety:** проверяют вредоносный запрос, вмешательство через данные и ошибки агента при обычном запросе. Этот notebook не реализует отдельный safety benchmark.

**Сравнение результатов:** фиксируйте модель, настройки генерации, версию кода, задания, scorer, стоимость и число попыток. Публичный leaderboard может измерять другую систему и другие условия. Вместо подгонки под известные вопросы собирайте собственные примеры, отражающие реальные задачи.

Основные тесты проекта проверяют корректность оценщика и формул без LLM. Этот notebook отдельно показывает результаты живой модели; его выводы не сохраняются в Git.